In [0]:
#to download file from the source, we're requesting this module
import urllib.request

#this is the location of the file in CloudFront for NYC TAXI DATA
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

#we're basically telling databricks who's the current user
user = spark.sql("SELECT current_user()").collect()[0][0]

#and we dynamically put the user's name in the f-string placeholder
#we're doing this to create a temporary/local copy of the file inside our databricks workspace
path = f"/Workspace/Users/{user}/yellow_tripdata_2024-01.parquet"

#now we download the file
urllib.request.urlretrieve(url, path)

#we're creating a dataframe df and having spark read it as a parquet file.
df = spark.read.parquet("file:" + path)

#displays the dataframe
display(df)

#displays the structure of the dataframe
df.printSchema()

#we're creating a database but dont throw an error if it already exists
spark.sql("create database if not exists demodb")

#Remove the old table if it exists
spark.sql("DROP TABLE IF EXISTS demodb.trip_data")

#we're writing the dataframe to a delta table and if it already exists then we ask it to overwrite
df.write.mode("overwrite").saveAsTable("demodb.trip_data")

spark.sql("SELECT * FROM demodb.trip_data LIMIT 10").show()